<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_2_Data_Science_Core/Month_05_Data_Cleaning_and_Preprocessing/Week_1_Handling_Missing_Data/Day_07_Week_1_Review_and_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 7: Week 1 Review and Missing Data Mini-Project

## Phase 2 | Month 5 | Week 1 | Day 7

**🎯 Goal:** Apply the full Week 1 toolkit to a messy real-world-style dataset.
**⏱️ Study Session:** ~2-3 hours

### Week 1 Topics Covered
1. ✓ Introduction to Data Cleaning — quality dimensions, profiling
2. ✓ Identifying and Exploring Missing Data — mechanisms (MCAR/MAR/MNAR)
3. ✓ Dropping Missing Values — listwise, pairwise, threshold
4. ✓ Mean/Median/Mode Imputation — global and group-wise
5. ✓ Advanced Imputation — KNN and Iterative Imputer
6. ✓ Handling Duplicates and Outliers — IQR, Z-score, Winsorise

## 📊 Mini-Project: Cleaning an NHIF (National Health Insurance Fund) Dataset

You will clean a simulated NHIF dataset containing registration errors, missing values, duplicates, and outliers — the type of challenge faced by NHIF data analysts.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2024)
N = 1500

def nhif_dataset(N, seed=2024):
    np.random.seed(seed)
    ages  = np.random.normal(38, 12, N).clip(18, 80).round(0)
    df = pd.DataFrame({
        'member_id':       list(range(1, N+1)) + [200, 500, 800],
        'name':            ['Member_'+str(i) for i in range(1, N+1)] + ['Member_200','Member_500','Member_800'],
        'age':             np.concatenate([ages, [38, 42, 55]]),
        'county':          np.concatenate([np.random.choice(
                               ['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret'],
                               N, p=[0.35,0.20,0.15,0.15,0.15]), ['Nairobi','Mombasa','Kisumu']]),
        'employment_type': np.concatenate([np.where(np.random.rand(N)<0.15, np.nan,
                               np.random.choice(['Formal','Informal','Self-employed'], N,
                                               p=[0.45,0.35,0.20])), [np.nan, np.nan, np.nan]]),
        'monthly_contrib': np.concatenate([np.where(np.random.rand(N)<0.20, np.nan,
                               np.random.choice([500,1000,1700], N, p=[0.3,0.5,0.2])),
                               [np.nan, np.nan, np.nan]]),
        'num_dependants':  np.concatenate([np.where(np.random.rand(N)<0.10, np.nan,
                               np.random.randint(0,7,N).astype(float)), [np.nan, np.nan, np.nan]]),
        'annual_claims_kes': np.concatenate([np.random.lognormal(9.5, 1.2, N-3),
                               [5_000_000, -100, 0], [0, 0, 0]]),  # outliers/errors
    })
    return df

df_raw = nhif_dataset(N)
print(f'Shape: {df_raw.shape}')
print(f'\nMissing:\n{df_raw.isnull().sum()}')
print(f'\nDuplicates: {df_raw.duplicated(subset="member_id").sum()}')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(2024)
N = 1500
ages = np.random.normal(38,12,N).clip(18,80).round(0)
df = pd.DataFrame({
    'member_id':       list(range(1,N+1))+[200,500,800],
    'age':             np.concatenate([ages,[38,42,55]]),
    'county':          np.concatenate([np.random.choice(['Nairobi','Mombasa','Kisumu','Nakuru','Eldoret'],N,p=[0.35,0.20,0.15,0.15,0.15]),['Nairobi','Mombasa','Kisumu']]),
    'employment_type': np.concatenate([np.where(np.random.rand(N)<0.15, np.nan,
                           np.random.choice(['Formal','Informal','Self-employed'],N,p=[0.45,0.35,0.20])),
                           [np.nan,np.nan,np.nan]]),
    'monthly_contrib': np.concatenate([np.where(np.random.rand(N)<0.20, np.nan,
                           np.random.choice([500,1000,1700],N,p=[0.3,0.5,0.2])),
                           [np.nan,np.nan,np.nan]]),
    'num_dependants':  np.concatenate([np.where(np.random.rand(N)<0.10, np.nan,
                           np.random.randint(0,7,N).astype(float)),[np.nan,np.nan,np.nan]]),
    'annual_claims_kes': np.concatenate([np.random.lognormal(9.5,1.2,N-3),[5_000_000,-100,0],[0,0,0]]),
})

# Step 1: Remove duplicates
df = df.drop_duplicates(subset='member_id', keep='last').reset_index(drop=True)
print(f'After dedup: {len(df)} rows')

# Step 2: Fix invalid values
df.loc[df['annual_claims_kes'] < 0, 'annual_claims_kes'] = np.nan

# Step 3: Winsorise outlier claims
q99 = df['annual_claims_kes'].quantile(0.99)
df['annual_claims_kes'] = df['annual_claims_kes'].clip(upper=q99)

# Step 4: Impute categorical with mode
df['employment_type'] = df.groupby('county')['employment_type'].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 'Formal')
)

# Step 5: KNN impute numeric
num_cols = ['num_dependants','monthly_contrib']
knn = KNNImputer(n_neighbors=5)
df[num_cols] = knn.fit_transform(df[num_cols])
df['num_dependants'] = df['num_dependants'].round(0)

print(f'\nMissing after cleaning:\n{df.isnull().sum()}')
print(f'\nCleaning complete! Final shape: {df.shape}')

## 📋 Week 1 Summary

| Problem | Technique | pandas / sklearn |
|---------|-----------|------------------|
| Missing numeric | Mean/Median | `SimpleImputer(strategy='mean')` |
| Missing categorical | Mode | `SimpleImputer(strategy='most_frequent')` |
| Missing (correlated) | KNN / MICE | `KNNImputer`, `IterativeImputer` |
| Duplicates | Drop/merge | `drop_duplicates()` |
| Outliers (error) | Drop / NaN | Conditional masking |
| Outliers (extreme) | Cap/Winsorise | `.clip(lo, hi)` |
| Missing at scale | Group-wise | `groupby().transform()` |

## 🚀 Next Steps
**Week 2: Feature Scaling and Transformation** — normalise your features for ML algorithms.

### 🌙 Weekend Challenge
Download the Kenya Open Data portal household survey CSV and apply a full Week 1 cleaning pipeline.